In [2]:
# filter dataset
!python3 MT-Preparation/filtering/filter.py ./en-zh.en ./en-zh.zh en zh

Dataframe shape (rows, columns): (231267, 2)
--- Rows with Empty Cells Deleted	--> Rows: 231267
--- Duplicates Deleted			--> Rows: 229646
--- Source-Copied Rows Deleted		--> Rows: 229640
--- Too Long Source/Target Deleted	--> Rows: 224743
--- HTML Removed			--> Rows: 224743
--- Rows will remain true-cased		--> Rows: 224743
--- Rows with Empty Cells Deleted	--> Rows: 224743
--- Source Saved: ./en-zh.en-filtered.en
--- Target Saved: ./en-zh.zh-filtered.zh


In [ ]:
import spacy

# Load English model
nlp = spacy.load("en_core_web_sm")

def tag_file(input_file, output_file):
    with open(input_file, "r", encoding="utf-8") as infile, \
         open(output_file, "w", encoding="utf-8") as outfile:
        # Process sentences in batches for efficiency
        sentences = [line.strip() for line in infile]
        for doc in nlp.pipe(sentences, disable=["ner", "lemmatizer"]):  
            tagged_line = " ".join(f"{token.text}_{token.pos_}" for token in doc)
            outfile.write(tagged_line + "\n")

# Tag your English file
tag_file("en-zh.en", "en-zh_tagged.en")

/Users/ziyi03/Desktop/CS4248/myenv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [6]:
!pip install sentencepiece

     |████████████████████████████████| 1.2 MB 4.5 MB/s eta 0:00:01


In [29]:
# train a sentencepiece model for subwording
!python3 MT-Preparation/subwording/1-train_unigram.py ./en-zh_tagged.en ./en-zh.zh-filtered.zh 

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=./en-zh_tagged.en --model_prefix=source --vocab_size=10000 --hard_vocab_limit=false --split_digits=true --user_defined_symbols=__SEP__,_VERB,_NOUN,_PRON,_ADV,_PUNCT,_DET,_AUX,_ADJ,_PART,_ADP,_CCONJ,_PROPN
sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./en-zh_tagged.en
  input_format: 
  model_prefix: source
  model_type: UNIGRAM
  vocab_size: 10000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: __SEP__
  user_defined_symbols: _VERB
  user_defined_symbols

In [30]:
# subword the dataset
!python3 MT-Preparation/subwording/2-subword.py source.model target.model ./en-zh_tagged.en ./en-zh.zh-filtered.zh

Source Model: source.model
Target Model: target.model
Source Dataset: ./en-zh_tagged.en
Target Dataset: ./en-zh.zh-filtered.zh
Done subwording the source file! Output: ./en-zh_tagged.en.subword
Done subwording the target file! Output: ./en-zh.zh-filtered.zh.subword


In [22]:
# first 3 lines before subwording
!head -n 3 ./en-zh_tagged.en && echo "-----" && head -n 3 ./en-zh.zh-filtered.zh

en_VERB
Thank_VERB you_PRON so_ADV much_ADV ,_PUNCT Chris_PROPN ._PUNCT And_CCONJ it_PRON 's_AUX truly_ADV a_DET great_ADJ honor_NOUN to_PART have_VERB the_DET opportunity_NOUN to_PART come_VERB to_ADP this_DET stage_NOUN twice_ADV ;_PUNCT I_PRON 'm_AUX extremely_ADV grateful_ADJ ._PUNCT
I_PRON have_AUX been_AUX blown_VERB away_ADV by_ADP this_DET conference_NOUN ,_PUNCT and_CCONJ I_PRON want_VERB to_PART thank_VERB all_PRON of_ADP you_PRON for_ADP the_DET many_ADJ nice_ADJ comments_NOUN about_ADP what_PRON I_PRON had_AUX to_PART say_VERB the_DET other_ADJ night_NOUN ._PUNCT
-----
zh
非常谢谢，克里斯。的确非常荣幸 能有第二次站在这个台上的机会，我真是非常感激。
这个会议真是让我感到惊叹不已，我还要谢谢你们留下的 关于我上次演讲的精彩评论


In [31]:
# first 3 lines after subwording
!head -n 3 ./en-zh_tagged.en.subword && echo "---" && head -n 3 ./en-zh.zh-filtered.zh.subword 

▁en_VERB
▁Thank_VERB ▁you_PRON ▁so_ADV ▁much_ADV ▁,_PUNCT ▁Chris_PROPN ▁._PUNCT ▁And_CCONJ ▁it_PRON ▁'s_AUX ▁truly_ADV ▁a_DET ▁great_ADJ ▁honor_NOUN ▁to_PART ▁have_VERB ▁the_DET ▁opportunity_NOUN ▁to_PART ▁come_VERB ▁to_ADP ▁this_DET ▁stage_NOUN ▁twice_ADV ▁;_PUNCT ▁I_PRON ▁'m_AUX ▁extremely_ADV ▁grateful_ADJ ▁._PUNCT
▁I_PRON ▁have_AUX ▁been_AUX ▁blown_VERB ▁away_ADV ▁by_ADP ▁this_DET ▁conference_NOUN ▁,_PUNCT ▁and_CCONJ ▁I_PRON ▁want_VERB ▁to_PART ▁thank_VERB ▁all_PRON ▁of_ADP ▁you_PRON ▁for_ADP ▁the_DET ▁many_ADJ ▁nice_ADJ ▁comments_NOUN ▁about_ADP ▁what_PRON ▁I_PRON ▁had_AUX ▁to_PART ▁say_VERB ▁the_DET ▁other_ADJ ▁night_NOUN ▁._PUNCT
---
▁ z h
▁非常 谢谢 , 克里斯 。 的确 非常 荣幸 ▁能 有 第二次 站在 这个 台上 的机会 , 我 真是 非常 感激 。
▁这个 会议 真是 让我 感到 惊 叹 不 已 , 我 还要 谢谢你们 留下 的 ▁关于 我 上 次 演讲 的 精彩 评论


In [32]:
# split the dataset into training set, development set, and test set
# Development and test sets should be between 1000 and 5000 segments (here we chose 200)
!python3 MT-Preparation/train_dev_split/train_dev_test_split.py 2000 2000 ./en-zh_tagged.en.subword ./en-zh.zh-filtered.zh.subword

Dataframe shape: (231267, 2)
--- Empty Cells Deleted --> Rows: 224743
--- Wrote Files
Done!
Output files
./en-zh_tagged.en.subword.train
./en-zh.zh-filtered.zh.subword.train
./en-zh_tagged.en.subword.dev
./en-zh.zh-filtered.zh.subword.dev
./en-zh_tagged.en.subword.test
./en-zh.zh-filtered.zh.subword.test


In [33]:
!wc -l ./*.subword.*

    2000 ./en-zh.zh-filtered.zh.subword.dev
    2000 ./en-zh.zh-filtered.zh.subword.test
  220743 ./en-zh.zh-filtered.zh.subword.train
    2000 ./en-zh_tagged.en.subword.dev
    2000 ./en-zh_tagged.en.subword.test
  220743 ./en-zh_tagged.en.subword.train
  449486 total


In [34]:
# check the first and last line from each dataset
!echo "---First line---"
!head -n 1 ./*.{train,dev,test}

!echo -e "\n---Last line---"
!tail -n 1 ./*.{train,dev,test}

---First line---
==> ./en-zh.zh-filtered.zh.subword.train <==
▁ z h

==> ./en-zh_tagged.en.subword.train <==
▁en_VERB

==> ./en-zh.zh-filtered.zh.subword.dev <==
▁所以 这两 样东西 是 联合 起来 的 。 ▁其实 就是 你的 受 教育 程度 和 周围 邻居 的 类型 , ▁我们 一会儿 再 具体 的 谈 一 谈 。

==> ./en-zh_tagged.en.subword.dev <==
▁It_PRON ▁'s_AUX ▁not_PART ▁the_DET ▁way_NOUN ▁they_PRON ▁occur_VERB ▁in_ADP ▁the_DET ▁film_NOUN ▁,_PUNCT ▁because_SCONJ ▁they_PRON ▁are_AUX ▁part_NOUN ▁of_ADP ▁much_ADV ▁larger_ADJ ▁scenes_NOUN ▁._PUNCT

==> ./en-zh.zh-filtered.zh.subword.test <==
▁在 捷 克斯 洛 伐 克 , 东 德 ▁ 爱 沙 尼亚 , 拉 脱 维 亚 , 立 陶 宛 , ▁ 马 里 , 马 达 加 斯 加 , ▁ 波 兰 , 菲 律 宾 , ▁ 塞 尔 维 亚 , 斯 洛 维 尼亚 的 独裁 政府 , 我可以 继续 , ▁还有 现在 的 突 尼 斯 和 埃及 。

==> ./en-zh_tagged.en.subword.test <==
▁He_PRON ▁learned_VERB ▁,_PUNCT ▁from_ADP ▁watching_VERB ▁this_DET ▁kid_NOUN ▁who_PRON ▁he_PRON ▁eventually_ADV ▁hired_VERB ▁,_PUNCT ▁who_PRON ▁was_AUX ▁really_ADV ▁successful_ADJ ▁for_ADP ▁him_PRON ▁,_PUNCT ▁that_SCONJ ▁it_PRON ▁was_AUX ▁n't_PART ▁swinging_VERB ▁for_ADP ▁the_DET ▁fe